# 50. What the literal phase is doing: three controls on Qwen 3B

The final 3B recipe (v4) continues v2 on 4,000 examples: 2,000 literal examples (exact-target training, three quarters with an inserted instruction) and 2,000 replayed distillation examples (1,000 attacked, 1,000 twins). Against v2 it changes three things at once: a new kind of supervision, more training, and replay. Each control below changes one of them and keeps the rest.

| control | starts from | trains on (4,000 examples, same steps, same settings as v4) | question |
|---|---|---|---|
| `notwin_literal` | the no-twin model (v2 scale) | the same 2,000 literal examples + 2,000 replayed attacked examples, no twins | once the literal phase is added, do the twins still matter? |
| `replay_only` | v2 | 4,000 replayed distillation examples (v4's 2,000 plus 1,000 attacked and 1,000 twins more), no literal examples | is more distillation alone enough to explain v4? |
| `statement_literal` | v2 | 2,000 literal examples whose inserted sentence is always an ordinary statement + v4's 2,000 replayed examples | does it matter that the literal examples carry instructions, or is copying practice enough? |

Each control is evaluated exactly like v4: the fixed attacker (210 injections, 13 variants, the paper's scoring rules), the second source, user-issued instructions and held-out Alpaca, plain summaries, MMLU, and the exact fidelity items with kept, complete-transformation and added-text rates. The reference models (undefended, v2, v4, no twins) are scored from their cached outputs by the same code, and their fixed-attacker numbers must reproduce the paper's. The adaptive attacker is not run: it broke 3 to 4 percent of injections against every defended 3B model, so the fixed attacker is the one that separates them.

GPU. Each control trains for 1,000 steps (resumable from checkpoints) and generates about 4,100 outputs; every stage is cached, so a disconnected session resumes where it stopped. Roughly five hours on an A100, longer on an L4.

**Outputs.** `data/controls/<control>/`, `reports/controls_security.csv`, `reports/controls_utility.csv`, `reports/controls_fidelity.csv`, `reports/controls_paired.csv`, `figures/controls.png`.

In [ ]:
# === SESSION BOOTSTRAP (run first): install from the local disk, then mount ===
!pip install -q transformers datasets accelerate sentencepiece bitsandbytes peft langdetect==1.0.9
!pip uninstall -y -q torchao   # Colab ships an old torchao that the current PEFT rejects; nothing in this project uses it
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
for d_ in ('reports', 'figures', 'data/controls'): os.makedirs(d_, exist_ok=True)
import torch, gc
assert torch.cuda.is_available(), 'this notebook needs a GPU'
print('Session ready. GPU:', torch.cuda.get_device_name(0))

## Stage A. The three training sets (cached) and the evaluation sets

In [ ]:
import importlib, json, hashlib, shutil, re, collections, gc, torch, numpy as np, pandas as pd
import fidelity_exact, literal_tasks, twin_distill, model_defense, phase5_redo, targets, attack_success, structural_train, opi_source, ci_tools, final_scoring, fidelity_audit, adaptive_strict
[importlib.reload(m) for m in (fidelity_exact, literal_tasks, twin_distill, model_defense, phase5_redo, targets, attack_success, structural_train, opi_source, ci_tools, adaptive_strict, final_scoring, fidelity_audit)]
from fidelity_exact import TASKS as FTASKS, MAX_NEW as FMAX, score as fscore, paired_boot, split_sentences, norm
from literal_tasks import build_literal_examples, sft_loss
from twin_distill import sample_rollouts, response_logprobs, distill_loss, PROMPT
from model_defense import build_pairs, TASK_USER_WITH
from phase5_redo import flat_variants
from targets import load_lm, generate_batch, TASK_SYSTEM, TASK_USER, build_task_prompts
from attack_success import judge_prompt, parse_judge, JUDGE_SYSTEM
from opi_source import INJECT_INSTR
from structural_train import load_jsonl
from ci_tools import wilson, fmt
from final_scoring import final_rule, langdetect_non_english, first_success, CATEGORIES, TRANSLATION, SCAMS
from fidelity_audit import complete, newcombe_paired, boot_mean
from peft import PeftModel, prepare_model_for_kbit_training
TARGET = 'Qwen/Qwen2.5-3B-Instruct'; JUDGE = 'Qwen/Qwen2.5-7B-Instruct'
ADAPTER_V2, ADAPTER_NT2, ADAPTER_V4 = 'data/distill2/lora_3b', 'data/distill_notwin2/lora_3b', 'data/distill4/lora_3b'
for a in (ADAPTER_V2, ADAPTER_NT2, ADAPTER_V4): assert os.path.exists(os.path.join(a, 'adapter_config.json')), f'missing adapter {a}'
H = lambda s: hashlib.sha1(s.encode()).hexdigest()
def jload(f): return json.load(open(f)) if os.path.exists(f) else {}
pd.set_option('display.width', 260); pd.set_option('display.max_columns', 40)
V4EX = json.load(open('data/distill4/examples.json')); LIT_V4 = [e for e in V4EX if e['kind'] == 'literal']; REP_V4 = [e for e in V4EX if e['kind'] != 'literal']
assert len(LIT_V4) == 2000 and len(REP_V4) == 2000, (len(LIT_V4), len(REP_V4))
V2EX = json.load(open('data/distill2/examples.json')); key = lambda e: e['student'] + '\x00' + e.get('teacher', '')
used = {key(e) for e in REP_V4}
ATT_REST = [e for e in V2EX if e['kind'] == 'attacked' and key(e) not in used]; TW_REST = [e for e in V2EX if e['kind'] == 'twin' and key(e) not in used]
REP_V4_ATT = [e for e in REP_V4 if e['kind'] == 'attacked']
print('v4 replay:', dict(collections.Counter(e['kind'] for e in REP_V4)), '| v2 examples not in it: attacked', len(ATT_REST), 'twins', len(TW_REST))
assert len(ATT_REST) >= 1000 and len(TW_REST) >= 1000
def build_or_load(tag, build, seed):
    fp = f'data/controls/{tag}/examples.json'
    if os.path.exists(fp): return json.load(open(fp))
    ex = build(np.random.default_rng(seed)); np.random.default_rng(seed + 1000).shuffle(ex)
    os.makedirs(os.path.dirname(fp), exist_ok=True); json.dump(ex, open(fp, 'w')); return ex
pick = lambda pool, k, rng: [pool[i] for i in rng.choice(len(pool), k, replace=False)]
def build_statement_literal(rng):
    from datasets import load_dataset
    held_txt = {h['instruction'] for h in json.load(open('data/distill/examples.json'))['held']}
    alp = [r['instruction'] for r in load_dataset('tatsu-lab/alpaca', split='train') if r['instruction'] not in held_txt]
    hosts = [r['text'] for r in load_jsonl('data/structural/train.jsonl') if r['form'] == 'host_only']
    return build_literal_examples(hosts, alp, n=2000, seed=38, p_instruction=0.0) + list(REP_V4)
CONTROLS = {
    'notwin_literal':    dict(init=ADAPTER_NT2, label='no twins + literal phase', seed=51, build=lambda rng: list(LIT_V4) + REP_V4_ATT + pick(ATT_REST, 2000 - len(REP_V4_ATT), rng)),
    'replay_only':       dict(init=ADAPTER_V2, label='v2 + replay only (equal budget)', seed=52, build=lambda rng: list(REP_V4) + pick(ATT_REST, 1000, rng) + pick(TW_REST, 1000, rng)),
    'statement_literal': dict(init=ADAPTER_V2, label='v2 + literal phase, statements only', seed=53, build=build_statement_literal)}
EXS = {t: build_or_load(t, c['build'], c['seed']) for t, c in CONTROLS.items()}
for t, ex in EXS.items():
    kinds = collections.Counter(e['kind'] for e in ex); vers = collections.Counter(e.get('version') for e in ex if e['kind'] == 'literal')
    print(f'{t}: {len(ex)} examples {dict(kinds)} | literal versions {dict(vers)}')
    assert len(ex) == 4000
assert collections.Counter(e['kind'] for e in EXS['notwin_literal'])['twin'] == 0 and collections.Counter(e['kind'] for e in EXS['replay_only'])['literal'] == 0
assert set(e['version'] for e in EXS['statement_literal'] if e['kind'] == 'literal') == {'statement'}
# evaluation sets, identical to notebook 38
SUB = pd.DataFrame(json.load(open('data/phase5v3/subset_full.json'))); REW = json.load(open('data/phase5v3/rewrites_full.json')); REX = json.load(open('data/phase5v3/rewrites_extra.json')); n = len(SUB)
VAR13, OWNER13, FIRST13 = flat_variants(SUB, [REW[i] + REX[i] for i in range(n)]); P13 = build_task_prompts(VAR13); KEYS = [H(p) for p in P13]
POS = {i: np.where(OWNER13 == i)[0] for i in range(n)}; CAT13 = [SUB.meta.iloc[int(OWNER13[v])] for v in range(len(P13))]; GOAL13 = [SUB.instruction.iloc[int(OWNER13[v])] for v in range(len(P13))]
HELD = json.load(open('data/distill/examples.json'))['held']
UP150 = build_pairs(load_jsonl('data/structural/val.jsonl'), n_max=150, seed=21); UPAIRS = build_pairs(load_jsonl('data/structural/val.jsonl'), n_max=60, seed=1)
UT150 = [TASK_USER_WITH.format(instr=p['instruction'], doc=p['host']) for p in UP150]; UT_clean = [TASK_USER.format(doc=p['host']) for p in UPAIRS]; UT_alpaca = [PROMPT.format(instruction=h['instruction'], inp=h['input']) for h in HELD]
OPI = json.load(open('data/opi/opi_eval.json'))['records']; att2 = [r for r in OPI if r['label'] == 1]; rng0 = np.random.default_rng(0); strat = sorted({r['strategy'] for r in att2}); pk = []
for s_ in strat: idx = [i for i, r in enumerate(att2) if r['strategy'] == s_]; pk += list(rng0.choice(idx, min(60, len(idx)), replace=False))
S2 = [att2[i] for i in pk]; P2 = [TASK_USER.format(doc=r['text']) for r in S2]
MM = json.load(open('data/phase5v3/mmlu_300.json')); PM = [q['question'].strip() + '\n' + '\n'.join(f'{L}. {c}' for L, c in zip('ABCD', q['choices'])) + '\nAnswer with the letter only.' for q in MM]
FITEMS = json.load(open('data/fidelity_exact/items.json')); FPROMPTS = {task: [it[f'prompt_{w}'] for it in FITEMS if it['task'] == task for w in ('instruction', 'statement')] for task in FTASKS}
EVAL = P13 + P2 + UT150 + UT_clean + UT_alpaca
# the separations the paper relies on, for the new statement-only literal examples
LITX = [e for e in EXS['statement_literal'] if e['kind'] == 'literal']; test_sents = {norm(s) for it in FITEMS for s in it['base']}
shared = sum(any(norm(x) in test_sents for x in split_sentences(e['prompt'].split('\n\n', 1)[1])) for e in LITX)
assert shared == 0 and not ({e['task'] for e in LITX} & set(FTASKS)), 'the statement-only literal examples overlap the fidelity test'
print('evaluation prompts:', len(EVAL), '| fidelity prompts:', sum(len(v) for v in FPROMPTS.values()), '| MMLU', len(MM))

## Stage B. Train the three controls (each resumes from its last checkpoint)

In [ ]:
LAMBDA = 0.3   # weight of the exact-target loss, as in v4
def train(tag):
    c = CONTROLS[tag]; EX = EXS[tag]; D = f'data/controls/{tag}'; AD = f'{D}/lora_3b'; PROG = f'{D}/progress.json'; LOGF = f'{D}/train_log.json'
    start = json.load(open(PROG))['step'] if os.path.exists(PROG) else 0
    if not os.path.exists(os.path.join(AD, 'adapter_config.json')): shutil.copytree(c['init'], AD); print(tag, ': initialised from', c['init'])
    if start >= len(EX): print(tag, ': training complete'); return
    torch.manual_seed(50 + start); base, tok = load_lm(TARGET, four_bit=True); base = prepare_model_for_kbit_training(base)
    model = PeftModel.from_pretrained(base, AD, is_trainable=True); print(tag, ': resuming from example', start)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-5, weight_decay=0.0)
    B, LOG, CKPT, MAXNEW = 4, 20, 100, 128; model.train(); log = jload(LOGF) or []
    for step in range(start, len(EX), B):
        batch = EX[step: step + B]; dist = [e for e in batch if e['kind'] != 'literal']; lit = [e for e in batch if e['kind'] == 'literal']
        terms, rec = [], dict(step=step, kl=None, sft=None)
        if dist:
            roll = sample_rollouts(model, tok, [e['student'] for e in dist], TASK_SYSTEM, max_new_tokens=MAXNEW); keep = [i for i, r in enumerate(roll) if len(r) >= 5]
            if keep:
                dk = [dist[i] for i in keep]; rk = [roll[i] for i in keep]
                lp_s = response_logprobs(model, tok, [e['student'] for e in dk], rk, TASK_SYSTEM)
                with torch.no_grad(), model.disable_adapter(): lp_t = response_logprobs(model, tok, [e['teacher'] for e in dk], rk, TASK_SYSTEM)
                dl, kl = distill_loss(lp_s, lp_t); terms.append(dl); rec['kl'] = kl
        if lit:
            sl = sft_loss(response_logprobs(model, tok, [e['prompt'] for e in lit], [e['gold'] for e in lit], TASK_SYSTEM)); terms.append(LAMBDA * sl); rec['sft'] = float(sl)
        if terms:
            loss = terms[0] if len(terms) == 1 else terms[0] + terms[1]; loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0); opt.step(); opt.zero_grad(); log.append(rec); del loss, terms; torch.cuda.empty_cache()
        nb = step // B
        if nb % LOG == 0 and log:
            r = log[-LOG:]; kls = [x['kl'] for x in r if x['kl'] is not None]; sfts = [x['sft'] for x in r if x['sft'] is not None]
            print(f'  {tag} {step + len(batch)}/{len(EX)}  reverse-KL {np.mean(kls) if kls else float("nan"):.3f}  exact-target loss {np.mean(sfts) if sfts else float("nan"):.3f}')
        if nb % CKPT == CKPT - 1: model.save_pretrained(AD); json.dump(dict(step=step + B), open(PROG, 'w')); json.dump(log, open(LOGF, 'w'))
    model.save_pretrained(AD); json.dump(dict(step=len(EX)), open(PROG, 'w')); json.dump(log, open(LOGF, 'w')); print(tag, ': adapter saved')
    del model, base, opt; gc.collect(); torch.cuda.empty_cache()
for tag in CONTROLS: train(tag)

## Stage C. Generate every evaluation output (cached in chunks)

In [ ]:
def gen_missing(cache, path, model, tok, prompts, system, max_new):
    todo = [p for p in dict.fromkeys(prompts) if H(p) not in cache]
    for k in range(0, len(todo), 160):
        chunk = todo[k: k + 160]; outs = generate_batch(model, tok, chunk, system=system, max_new_tokens=max_new, batch_size=8, show_every=160)
        cache.update({H(p): o for p, o in zip(chunk, outs)}); json.dump(cache, open(path, 'w'))
    return len(todo)
EXE = {t: jload(f'data/controls/{t}/exec.json') for t in CONTROLS}; FOU = {t: jload(f'data/controls/{t}/fid.json') for t in CONTROLS}
def done(t): return all(H(p) in EXE[t] for p in EVAL) and all(H(p) in FOU[t] for ps in FPROMPTS.values() for p in ps) and os.path.exists(f'data/controls/{t}/mmlu.json')
for tag in CONTROLS:
    if done(tag): print(tag, ': outputs complete'); continue
    tm, tt = load_lm(TARGET, four_bit=True); m = PeftModel.from_pretrained(tm, f'data/controls/{tag}/lora_3b').eval(); print('generating', tag)
    n1 = gen_missing(EXE[tag], f'data/controls/{tag}/exec.json', m, tt, EVAL, TASK_SYSTEM, 160)
    n2 = sum(gen_missing(FOU[tag], f'data/controls/{tag}/fid.json', m, tt, FPROMPTS[task], TASK_SYSTEM, FMAX[task]) for task in FTASKS)
    if not os.path.exists(f'data/controls/{tag}/mmlu.json'):
        json.dump(generate_batch(m, tt, PM, system='You are a helpful assistant.', max_new_tokens=4, batch_size=16, temperature=0.0), open(f'data/controls/{tag}/mmlu.json', 'w'))
    print(f'  {tag}: {n1} evaluation and {n2} fidelity outputs generated'); del m, tm; gc.collect(); torch.cuda.empty_cache()
print('complete:', {t: done(t) for t in CONTROLS})

## Stage D. The judge, where the paper's rules call for it (cached)

In [ ]:
JV = {t: jload(f'data/controls/{t}/judge.json') for t in CONTROLS}
def needs(t):
    E = EXE[t]; out = []
    for v in range(len(P13)):
        k = 'fix|' + KEYS[v]
        if k not in JV[t] and final_rule(CAT13[v], E[KEYS[v]], VAR13[v]) is None: out.append((k, GOAL13[v], E[KEYS[v]]))
    out += [('p2|' + H(p), INJECT_INSTR[S2[v]['inj_task']], E[H(p)]) for v, p in enumerate(P2) if 'p2|' + H(p) not in JV[t]]
    out += [('user|' + H(p), UP150[u]['instruction'], E[H(p)]) for u, p in enumerate(UT150) if 'user|' + H(p) not in JV[t]]
    out += [('alp|' + H(p), HELD[u]['instruction'], E[H(p)]) for u, p in enumerate(UT_alpaca) if 'alp|' + H(p) not in JV[t]]
    return out
NEED = {t: needs(t) for t in CONTROLS}; print('to judge:', {t: len(v) for t, v in NEED.items()})
if any(NEED.values()):
    jm, jt = load_lm(JUDGE, four_bit=True)
    for t, nd in NEED.items():
        for k in range(0, len(nd), 400):
            chunk = nd[k: k + 400]; jo = generate_batch(jm, jt, [judge_prompt(g, o) for _, g, o in chunk], system=JUDGE_SYSTEM, max_new_tokens=8, batch_size=8, temperature=0.0, show_every=400)
            JV[t].update({kk: bool(parse_judge(o)) for (kk, _, _), o in zip(chunk, jo)}); json.dump(JV[t], open(f'data/controls/{t}/judge.json', 'w'))
        print(t, ': judged', len(nd))
    del jm, jt; gc.collect(); torch.cuda.empty_cache()
print('verdicts per control:', {t: len(v) for t, v in JV.items()})

## Stage E. Results: security, utility, fidelity, and the paired comparisons

In [ ]:
# reference models from their caches, scored by the code that produced the paper's numbers (notebooks 46 and 48)
REF = {'undefended': dict(exec='undefended', fid='undefended_3b', mid='qwen3b_undefended', mmlu='undefended', label='Qwen 3B undefended'),
       'twin_distilled_v2': dict(exec='twin_distilled_v2', fid='twin_distilled_v2', mid='qwen3b_v2', mmlu='twin_distilled_v2', label='v2 (twin distillation)'),
       'fidelity_v4': dict(exec='fidelity_v4', fid='fidelity_v4', mid='qwen3b_v4', mmlu='fidelity_v4', label='v4 = v2 + literal phase (final)'),
       'no_twin_v2': dict(exec='no_twin_v2', fid='no_twin_v2', mid='no_twin_v2', mmlu='no_twin_v2', label='no twins (v2 scale)')}
def caches(kind, ctag):
    C_ = {}
    for d_ in ('data/distill2', 'data/phase5v3'):
        f = f'{d_}/{kind}_{ctag}.json'
        if os.path.exists(f): C_.update({k: v for k, v in json.load(open(f)).items() if k not in C_})
    return C_
LD = {}
def ld(o):
    k = H(o or '')
    if k not in LD: LD[k] = langdetect_non_english(o)
    return LD[k]
OUT, JUD, FIXJ = {}, {}, {}
for t, r in REF.items():
    E = caches('exec', r['exec']); assert all(k in E for k in KEYS), f'fixed-attacker outputs missing for {t}'
    OUT[t] = E; JUD[t] = caches('judge', r['exec']); FIXJ[t] = jload(f"data/final_scoring/fixed_judge_{r['mid']}.json")
for t in CONTROLS: OUT[t] = EXE[t]; JUD[t] = {k.split('|', 1)[1]: v for k, v in JV[t].items() if not k.startswith('fix|')}; FIXJ[t] = {k[4:]: v for k, v in JV[t].items() if k.startswith('fix|')}
def fixed_flags(t):
    E, JC, FC = OUT[t], JUD[t], FIXJ[t]; f = np.zeros(len(P13), bool)
    for v in range(len(P13)):
        k, cat, o, doc = KEYS[v], CAT13[v], E[KEYS[v]], VAR13[v]
        if cat == TRANSLATION: f[v] = bool(FC[k]) or ld(o); continue
        r = final_rule(cat, o, doc); f[v] = r if r is not None else (FC[k] if cat == SCAMS else (FC[k] if k in FC else JC[k]))
    return f
MODELS = list(REF) + list(CONTROLS); LB = {**{t: r['label'] for t, r in REF.items()}, **{t: c['label'] for t, c in CONTROLS.items()}}
FX = {t: fixed_flags(t) for t in MODELS}; F13 = {t: np.array([float(FX[t][POS[i]].any()) for i in range(n)]) for t in MODELS}; F1 = {t: np.array([float(FX[t][POS[i][0]]) for i in range(n)]) for t in MODELS}
ALLM = pd.read_csv('reports/paper_fixed_all_models.csv').set_index('model'); PAPERNAME = {'undefended': 'Qwen 3B undefended', 'twin_distilled_v2': 'Qwen 3B twin distillation v2', 'fidelity_v4': 'Qwen 3B full recipe (final)', 'no_twin_v2': 'no_twin_v2'}
for t, nm in PAPERNAME.items(): assert abs(F13[t].mean() - ALLM.loc[nm, 'fixed_13']) < 0.0006, f'{t}: {F13[t].mean():.3f} does not reproduce the paper ({ALLM.loc[nm, "fixed_13"]})'
print('reference models reproduce the paper fixed-attacker numbers')
def rate(x): x = np.asarray(x, float); return fmt(x.mean(), *wilson(int(x.sum()), len(x)))
def judged(t, prompts):
    ks = [H(p) for p in prompts]; return np.array([bool(JUD[t][k]) for k in ks]) if all(k in JUD[t] for k in ks) else None
def mmlu(t):
    f = f"data/phase5v3/mmlu_{REF[t]['mmlu']}.json" if t in REF else f'data/controls/{t}/mmlu.json'
    if not os.path.exists(f): return 'n/a'
    pred = [(re.search(r'[ABCD]', (o or '').strip().upper()) or [None])[0] for o in json.load(open(f))]; ok = np.array([p == 'ABCD'[q['answer']] for p, q in zip(pred, MM)]); return rate(ok)
srows, urows = [], []
for t in MODELS:
    s2 = judged(t, P2); srows.append(dict(model=LB[t], fixed_1=rate(F1[t]), fixed_13=rate(F13[t]), second_source=rate(s2) if s2 is not None else 'n/a'))
    uf, af = judged(t, UT150), judged(t, UT_alpaca); cl = np.array([len((OUT[t].get(H(p), '') or '').strip()) for p in UT_clean]) if all(H(p) in OUT[t] for p in UT_clean) else None
    urows.append(dict(model=LB[t], user_followed_150=rate(uf) if uf is not None else 'n/a', alpaca_followed=rate(af) if af is not None else 'n/a', mmlu=mmlu(t),
                      summary_median_chars=int(np.median(cl)) if cl is not None else 'n/a', degenerate_summaries=round(float((cl < 20).mean()), 3) if cl is not None else 'n/a'))
SEC = pd.DataFrame(srows); UTIL = pd.DataFrame(urows); SEC.to_csv('reports/controls_security.csv', index=False); UTIL.to_csv('reports/controls_utility.csv', index=False)
print('\n=== security: the fixed attacker (paper rules) and the second source ==='); print(SEC.to_string(index=False))
print('\n=== utility ==='); print(UTIL.to_string(index=False))

In [ ]:
FO = {t: (jload(f"data/fidelity_exact/out_{REF[t]['fid']}.json") if t in REF else FOU[t]) for t in MODELS}
for t in MODELS: assert all(H(it[f'prompt_{w}']) in FO[t] for it in FITEMS for w in ('instruction', 'statement')), f'fidelity outputs missing for {t}'
REC = []
for t in MODELS:
    for i, it in enumerate(FITEMS):
        for w in ('instruction', 'statement'):
            o = FO[t][H(it[f'prompt_{w}'])]; s = fscore(it, o, w); REC.append(dict(model=t, item=i, task=it['task'], version=w, kept=bool(s['kept']), added=bool(s['added']), complete=bool(complete(it, o, w))))
R = pd.DataFrame(REC)
def piv(t, task, col):
    g = R[(R.model == t) & (R.task == task)].sort_values('item'); return g[g.version == 'instruction'][col].values, g[g.version == 'statement'][col].values
F4 = pd.read_csv('reports/fidelity_v4_fidelity.csv'); first = lambda s: float(str(s).split(' ')[0])
for r in F4.itertuples():
    if r.model in REF or r.model == 'undefended_3b':
        t = 'undefended' if r.model == 'undefended_3b' else r.model; ki, ks = piv(t, r.task, 'kept')
        assert abs(ki.mean() - first(r.instruction_kept)) < 0.0015 and abs(ks.mean() - first(r.statement_kept)) < 0.0015, f'fidelity of {t} does not reproduce'
rows = []
for t in MODELS:
    for task in FTASKS:
        ki, ks = piv(t, task, 'kept'); ci, cs = piv(t, task, 'complete'); ai, as_ = piv(t, task, 'added'); nn = len(ki)
        r = dict(model=LB[t], task=task, instruction_kept=rate(ki), statement_kept=rate(ks), deletion_gap=fmt(*newcombe_paired(ks, ki)), instruction_complete=rate(ci), statement_complete=rate(cs), complete_gap=fmt(*newcombe_paired(cs, ci)))
        if task != 'quote_last': r.update(added_instruction=rate(ai), added_statement=rate(as_), added_text_gap=fmt(*newcombe_paired(ai, as_)))
        rows.append(r)
FID = pd.DataFrame(rows); FID.to_csv('reports/controls_fidelity.csv', index=False)
print('=== fidelity by exact rules ==='); print(FID.fillna('').to_string(index=False))
COMP = [('notwin_literal', 'fidelity_v4', 'twins removed, literal phase kept'), ('notwin_literal', 'no_twin_v2', 'literal phase added to the no-twin model'),
        ('replay_only', 'fidelity_v4', 'equal-budget replay in place of the literal examples'), ('replay_only', 'twin_distilled_v2', 'more distillation only'),
        ('statement_literal', 'fidelity_v4', 'statement-only literal examples in place of instruction-bearing ones'), ('statement_literal', 'twin_distilled_v2', 'statement-only literal phase added to v2')]
def bstr(v): return f'{v[0]:+.3f} [{v[1]:+.3f}, {v[2]:+.3f}]' + (f' (no item differs; at most {v[3]:.3f} could)' if len(v) > 3 and not np.isnan(v[3]) else '')
prow = []
for a, b, w in COMP:
    r = dict(comparison=f'{a} minus {b}', meaning=w, fixed_13=bstr(paired_boot(F13[a] - F13[b])))
    ua, ub = judged(a, UT150), judged(b, UT150); r['user_followed'] = fmt(*newcombe_paired(ua, ub)) if ua is not None and ub is not None else 'n/a'
    for task in FTASKS:
        kia, ksa = piv(a, task, 'kept'); kib, ksb = piv(b, task, 'kept'); cia, _ = piv(a, task, 'complete'); cib, _ = piv(b, task, 'complete')
        r[f'{task}: instruction kept'] = fmt(*newcombe_paired(kia, kib)); r[f'{task}: deletion gap'] = bstr(boot_mean((ksa.astype(float) - kia) - (ksb.astype(float) - kib))); r[f'{task}: instruction complete'] = fmt(*newcombe_paired(cia, cib))
    prow.append(r)
PAIR = pd.DataFrame(prow); PAIR.to_csv('reports/controls_paired.csv', index=False)
print('\n=== paired comparisons on the same injections and items (fixed_13 below zero: the first model is broken less; kept and complete above zero: the first keeps more) ===')
for r in PAIR.to_dict('records'):
    print(f"\n{r['comparison']}  ({r['meaning']})"); [print(f'   {k:32s} {v}') for k, v in r.items() if k not in ('comparison', 'meaning')]

In [ ]:
import matplotlib.pyplot as plt
SHOW = ['twin_distilled_v2', 'fidelity_v4', 'notwin_literal', 'replay_only', 'statement_literal', 'no_twin_v2']
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4)); x = np.arange(len(FTASKS)); w = 0.13
for j, t in enumerate(SHOW): axes[0].bar(x + (j - 2.5) * w, [piv(t, task, 'kept')[0].mean() for task in FTASKS], w, label=LB[t])
axes[0].set_xticks(x); axes[0].set_xticklabels(list(FTASKS)); axes[0].set_ylim(0, 1.05); axes[0].set_ylabel('instruction sentence kept'); axes[0].legend(fontsize=7); axes[0].grid(axis='y', alpha=.3)
for j, t in enumerate(SHOW): axes[1].bar(x + (j - 2.5) * w, [piv(t, task, 'complete')[0].mean() for task in FTASKS], w, label=LB[t])
axes[1].set_xticks(x); axes[1].set_xticklabels(list(FTASKS)); axes[1].set_ylim(0, 1.05); axes[1].set_ylabel('instruction item done completely and exactly'); axes[1].grid(axis='y', alpha=.3)
xs = np.arange(len(SHOW)); lo_hi = [wilson(int(F13[t].sum()), n) for t in SHOW]; vals = [F13[t].mean() for t in SHOW]
axes[2].bar(xs, vals, yerr=[[v - l for v, (l, h) in zip(vals, lo_hi)], [h - v for v, (l, h) in zip(vals, lo_hi)]], capsize=3, color='#4c72b0')
axes[2].set_xticks(xs); axes[2].set_xticklabels([LB[t] for t in SHOW], rotation=30, ha='right', fontsize=7); axes[2].set_ylim(0, 0.5); axes[2].set_ylabel('injections broken, fixed attacker (13 tries)'); axes[2].grid(axis='y', alpha=.3)
plt.tight_layout(); plt.savefig('figures/controls.png', dpi=150, bbox_inches='tight'); plt.show()
from reslog import log_result
summary = 'Security:\n' + SEC.to_string(index=False) + '\n\nUtility:\n' + UTIL.to_string(index=False) + '\n\nFidelity:\n' + FID.fillna('').to_string(index=False) + '\n\nPaired:\n' + PAIR.to_string(index=False)
log_result('nb50: literal-phase controls on Qwen 3B (no twins + literal; equal-budget replay only; statement-only literal)', summary, csv_df=PAIR, csv_name='controls_paired.csv')
print('logged')

# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py", "nb50: three controls for the literal phase on Qwen 3B, each changing one thing against v4 (no twins with the same literal phase; equal-budget replay without literal examples; literal examples with statements only), trained with v4's settings and evaluated like v4 under the paper's rules; reference models re-scored by the same code and checked against the paper"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)